<a href="https://colab.research.google.com/github/julschleinitz/ai4chemistry-bootcamp/blob/main/tutorials/learned-representations.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 7: Learned Representations — unsupervised structure and transfer learning

Hands-on companion to **Lecture 7 · Learned Representations**.

In the lecture we argued three things. This notebook makes you test all three on
the same dataset, so you leave with numbers rather than impressions.

1. **Unsupervised methods find real structure — but a 2-D map is a picture of your
   fingerprint, not of chemistry.**
2. **A pre-trained encoder gives you a representation for free.** No labels, no
   training: download weights, push molecules through, keep the vectors.
3. **Pre-training is a prior.** It is worth a lot when you have ~100 labels and
   close to nothing when you have thousands — and the crossover is something you
   measure, not something you look up.

## Learning goals

- Featurize a real dataset with ECFP4 and inspect it with PCA, UMAP and clustering
- See for yourself that neighbour embeddings invent clusters in pure noise
- Count Bemis–Murcko scaffolds and understand why a **scaffold split** is not optional
- Extract frozen embeddings from a pre-trained chemical language model (ChemBERTa)
- Compare a hand-crafted and a learned representation *without training anything*
- Run the three-way head-to-head: ECFP + random forest vs frozen embeddings vs both
- **Measure the learning-curve crossover on your own data** — the key result
- Find the activity-cliff pairs that break every smooth representation

## What this notebook deliberately does not do

We never update the encoder's weights. Everything here is the **frozen** setting
from the lecture's "three knobs" slide — because it is the cheapest, it is the
right first experiment, and it is impossible to fool yourself with. Gradient
fine-tuning is Thursday of week 2 (*Fine-tuning ChemBERTa*).

---

## References & tools used

- **Lipophilicity** (MoleculeNet) — 4,200 compounds with experimental octanol/water
  distribution coefficients (logD at pH 7.4), curated from ChEMBL.
  Wu, Z. *et al.* "MoleculeNet: a benchmark for molecular machine learning."
  *Chem. Sci.* **2018**, 9, 513. DOI 10.1039/C7SC02664A
- **ChemBERTa-2** — Ahmad, W.; Simon, E.; Chithrananda, S.; Grand, G.; Ramsundar, B.
  "ChemBERTa-2: Towards Chemical Foundation Models." arXiv:2209.01712 (2022).
  We use the `DeepChem/ChemBERTa-77M-MTR` checkpoint (77 M PubChem SMILES,
  multi-task-regression pre-training).
- **ECFP / Morgan fingerprints** — Rogers, D.; Hahn, M. *J. Chem. Inf. Model.*
  **2010**, 50, 742. DOI 10.1021/ci100050t
- **Butina clustering** — Butina, D. *J. Chem. Inf. Comput. Sci.* **1999**, 39, 747.
- **Bemis–Murcko scaffolds** — Bemis, G. W.; Murcko, M. A. *J. Med. Chem.* **1996**,
  39, 2887.
- **Why deep models often lose to fingerprints** — Deng, J. *et al.* *Nat. Commun.*
  **2023**, 14, 6395 · Praski, M.; Adamczyk, J.; Czech, W. arXiv:2508.06199 (2025).
- **How to misread t-SNE** — Wattenberg, Viégas & Johnson, *Distill* 2016,
  https://distill.pub/2016/misread-tsne/

Software: RDKit, scikit-learn, UMAP, HuggingFace `transformers`, PyTorch, pandas,
matplotlib.

---

## 0 · Setup

Run once. Colab already has PyTorch, pandas, scikit-learn and matplotlib; we add
RDKit, `transformers` and `umap-learn`.

In [ ]:
# --- Cell: dependency installation (Colab) ---
# Safe to re-run: pip is a no-op when the packages are already present.
!pip -q install rdkit transformers umap-learn > /dev/null
print("dependencies ready")

In [ ]:
# --- Cell: imports, seed and compute device ---
import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import rdFingerprintGenerator, Draw
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.ML.Cluster import Butina

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

import torch

RDLogger.DisableLog("rdApp.*")          # RDKit is chatty about sanitization
SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE)

# House palette, so the plots match the lecture slides.
TEAL, ORANGE, PURPLE, GRAY, RED = "#156082", "#E97132", "#A02B93", "#6E7480", "#C0392B"
plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.grid": True,
                     "grid.alpha": 0.25, "axes.spines.top": False,
                     "axes.spines.right": False})

In [ ]:
# --- Cell: locate the dataset, whether on Colab or running locally ---
# Order of preference:
#   1. a local copy next to the notebook (fastest, works offline)
#   2. the bootcamp repo on GitHub  (what Colab will normally use)
#   3. the MoleculeNet original on the DeepChem S3 bucket (last resort)
LOCAL_CANDIDATES = [
    "learned-representations/lipophilicity.csv",
    "lipophilicity.csv",
    "../learned-representations/lipophilicity.csv",
]
REMOTE_CANDIDATES = [
    "https://raw.githubusercontent.com/julschleinitz/ai4chemistry-bootcamp/"
    "main/tutorials/learned-representations/lipophilicity.csv",
    "https://deepchemdata.s3.us-west-1.amazonaws.com/datasets/Lipophilicity.csv",
]


def load_lipophilicity():
    for path in LOCAL_CANDIDATES:
        if os.path.exists(path):
            print("loaded local copy:", path)
            return pd.read_csv(path)
    for url in REMOTE_CANDIDATES:
        try:
            df = pd.read_csv(url)
            print("downloaded:", url)
            return df
        except Exception as exc:                       # noqa: BLE001
            print("could not fetch", url, "->", type(exc).__name__)
    raise RuntimeError("Lipophilicity.csv not found — see LOCAL_CANDIDATES above.")


raw = load_lipophilicity()
print(raw.shape)
raw.head()

---
## 1 · The dataset

**Lipophilicity** from MoleculeNet: 4,200 molecules with a measured logD at pH 7.4.

Why this dataset for this tutorial:

- It is a **regression** task, so the learning curve is easy to read (RMSE in log units).
- It is *just* big enough to bracket the crossover we care about — a few hundred to
  a few thousand labelled molecules is exactly the interesting regime.
- Lipophilicity is a whole-molecule, fairly smooth property. That makes it a
  **friendly** case for learned representations. Keep that in mind at the end:
  if pre-training struggles here, it will not do better on a spiky assay endpoint.

The columns are `CMPD_CHEMBLID`, `exp` (the logD value) and `smiles`.

In [ ]:
# --- Exercise 1: tidy the dataframe and look at the label distribution ---
# your code here!
#
# 1. Build `df` from `raw` with just two columns: "smiles" and "y" (= the `exp`
#    column). Drop rows whose SMILES fail to parse with Chem.MolFromSmiles.
# 2. Add a "mol" column holding the RDKit Mol objects (you will reuse them a lot).
# 3. Print the number of molecules and y.describe().
# 4. Plot a histogram of y.
#
# Hint: df["mol"] = df["smiles"].apply(Chem.MolFromSmiles), then drop the Nones.

raise NotImplementedError

---
## 2 · Part A — unsupervised structure

No labels are used anywhere in this section. We are asking what the *shape* of the
data looks like, which is something we could do for millions of molecules if we
wanted to, because it costs nothing but compute.

In [ ]:
# --- Exercise 2: featurize with ECFP4 ---
# your code here!
#
# 1. Create a Morgan fingerprint generator with radius=2 and fpSize=2048 using
#    rdFingerprintGenerator.GetMorganGenerator(...).
#    (radius 2 over a 2048-bit vector is what everyone means by "ECFP4".)
# 2. Build `X_ecfp`, a float32 numpy array of shape (n_molecules, 2048), using
#    gen.GetFingerprintAsNumPy(mol) for each molecule.
# 3. Also keep `fps_bv`, the list of RDKit ExplicitBitVect objects
#    (gen.GetFingerprint(mol)) — Tanimoto similarity needs these, not the arrays.
# 4. Print the shape of X_ecfp and the mean number of bits set per molecule.

raise NotImplementedError

### 2.1 PCA — the honest baseline

PCA is a rotation. It is linear, deterministic and invertible, and the scree plot
tells you exactly how much you are throwing away when you look at two components.

In [ ]:
# --- Exercise 3: PCA on the fingerprints ---
# your code here!
#
# 1. Fit PCA(n_components=50) on X_ecfp and transform it -> `Z_pca`.
# 2. Left panel: scatter Z_pca[:, 0] vs Z_pca[:, 1], coloured by df["y"],
#    with a colorbar. s=6, alpha=0.6 looks about right for 4k points.
# 3. Right panel: bar chart of the first 20 explained_variance_ratio_ values.
# 4. Print the cumulative variance explained by 2 and by 50 components.
#
# Question to answer in your head before you run it: how much of a 2048-bit
# fingerprint do you expect two linear components to capture?

raise NotImplementedError

**Read the numbers, not the picture.** On sparse binary fingerprints two components
typically capture only a few percent of the variance — far less than the
"2 components ≈ 60%" you would get on a handful of physicochemical descriptors.
Sparse bit vectors are near-orthogonal by construction, so there simply is no
low-dimensional *linear* structure to find. That is the motivation for nonlinear
neighbour embeddings.

### 2.2 t-SNE — and the control experiment nobody runs

In [ ]:
# --- Exercise 4: a chemical space map, plus the noise control ---
# your code here!
#
# Make a figure with three panels:
#   (a) t-SNE of the first 50 PCs of X_ecfp, coloured by logD.
#       Use TSNE(n_components=2, init="pca", perplexity=30, random_state=SEED)
#       and fit it on Z_pca (running t-SNE on the raw 2048 bits is slow and no better).
#   (b) The same t-SNE, but with perplexity=5. Same data, different picture.
#   (c) THE CONTROL: t-SNE of a matrix of *uniform random numbers* with the same
#       shape as Z_pca. No structure exists in this data at all.
#
# Then look at panel (c) and ask yourself whether you would have believed those
# clusters if someone had put them in a paper.

raise NotImplementedError

Three habits to take away from that figure:

1. **Never read clusters off a map.** If clustering matters to your argument, run a
   clustering algorithm in the original space and use the map only to display it.
2. **Report your hyperparameters.** A t-SNE without a stated perplexity, or a UMAP
   without `n_neighbors` and `min_dist`, is not reproducible.
3. **Run the noise control once in your life** so you never forget what it looks like.

### 2.3 Butina clustering — structure you can defend

In [ ]:
# --- Exercise 5: Butina clustering across a range of cut-offs ---
# your code here!
#
# 1. Write tanimoto_dists(fps) that returns the condensed lower-triangle distance
#    list RDKit's Butina wants:
#        for i in 1..n-1:  1 - DataStructs.BulkTanimotoSimilarity(fps[i], fps[:i])
#    Flatten those into one long list.
# 2. For cutoff in [0.3, 0.4, 0.5, 0.6, 0.7], call
#        Butina.ClusterData(dists, n, cutoff, isDistData=True)
#    and record: number of clusters, size of the largest, and the number of
#    singletons.
# 3. Print the table. Which cut-off would you use to pick a diverse subset?
#
# Note: this is O(n^2). On all 4,200 molecules that is 8.8 M pairs and a few
# hundred MB, so cluster a random subsample of CLUSTER_N = 2500 instead — the
# conclusion is identical and it runs in well under a minute.

raise NotImplementedError

### 2.4 Bemis–Murcko scaffolds — why random splits lie

In the lecture: 32 frameworks cover half of all known drugs. Let us see how
concentrated *this* dataset is.

In [ ]:
# --- Exercise 6: scaffold counts and the cumulative coverage curve ---
# your code here!
#
# 1. Add a "scaffold" column using
#        MurckoScaffold.MurckoScaffoldSmiles(mol=m, includeChirality=False)
#    (wrap it in try/except and fall back to "" for the odd molecule that fails).
# 2. Print how many distinct scaffolds there are and the size of the 5 largest groups.
# 3. Plot the cumulative fraction of molecules covered as you add scaffolds in
#    decreasing size order. Mark how many scaffolds it takes to cover 50%.
#
# This is the same figure as the Bemis-Murcko slide, computed on your own data.

raise NotImplementedError

---
## 3 · Part B — a learned representation, for free

Now the other half of the lecture. We download a chemical language model that
someone else pre-trained on 77 million PubChem SMILES, push our 4,200 molecules
through it, and keep the vectors.

**We do not train anything.** No labels are involved. This is the "frozen" column
of the three-knobs slide, and it takes about a minute.

In [ ]:
# --- Exercise 7: frozen ChemBERTa embeddings ---
# your code here!
#
# 1. Load the tokenizer and model:
#        from transformers import AutoTokenizer, AutoModel
#        MODEL_ID = "DeepChem/ChemBERTa-77M-MTR"
#    Put the model on DEVICE and call .eval(). (You will see a warning that some
#    checkpoint weights were not used — that is the regression head we are
#    deliberately throwing away. Exactly the "keep the encoder" slide.)
# 2. Write embed(smiles_list, batch_size=64) that, under torch.no_grad():
#      - tokenizes a batch with padding=True, truncation=True, max_length=256
#      - runs the model
#      - MEAN-POOLS last_hidden_state over the tokens, weighting by attention_mask
#        so padding does not contribute
#      - returns a float32 numpy array
# 3. Build `X_bert` for all of df["smiles"] and print its shape.
#
# Mean pooling with a mask:
#     h = out.last_hidden_state                       # (B, T, H)
#     m = mask.unsqueeze(-1).float()                  # (B, T, 1)
#     pooled = (h * m).sum(1) / m.sum(1).clamp(min=1e-9)

raise NotImplementedError

### 3.1 Are the learned vectors *better*? Ask without training anything

Here is a cheap, training-free way to compare two representations. For each
molecule, find its **k nearest neighbours** in representation space and measure how
different their labels are. A representation in which neighbours share a property
is a representation a model can exploit.

$$\text{neighbourhood roughness} = \frac{1}{N}\sum_i \frac{1}{k}\sum_{j \in \mathcal{N}_k(i)} |y_i - y_j|$$

Lower is better. Compare it against the roughness you get from *random* pairs —
that is the number to beat.

In [ ]:
# --- Exercise 8: neighbourhood roughness, ECFP vs ChemBERTa ---
# your code here!
#
# 1. Write roughness(M, y, k=5, metric="euclidean") that:
#      - fits NearestNeighbors(n_neighbors=k+1, metric=metric) on M
#      - for every point, takes its k neighbours (excluding itself, i.e. drop
#        column 0 of the returned indices)
#      - returns the mean |y_i - y_j| over all those pairs
# 2. Compute it for:
#      - ECFP4 with metric="jaccard"   (Jaccard distance == 1 - Tanimoto).
#        Pass X_ecfp.astype(bool) — sklearn wants booleans for this metric.
#      - ChemBERTa embeddings, standardized, with metric="euclidean"
#      - a random baseline: mean |y_i - y_j| over randomly paired molecules
# 3. Print all three. Which representation puts chemically similar molecules
#    closer together *in the sense that matters for this label*?

raise NotImplementedError

In [ ]:
# --- Exercise 9 (optional, but it makes a nice slide): the two maps side by side ---
# your code here!
#
# Run UMAP on both representations and plot them next to each other, coloured by
# logD:
#     import umap
#     reducer = umap.UMAP(n_neighbors=25, min_dist=0.1, random_state=SEED, metric=...)
# Use metric="jaccard" for ECFP and metric="euclidean" for the standardized
# ChemBERTa embeddings.
#
# Do the two maps tell the same story? Remember the caveats from Exercise 4 before
# you answer.

raise NotImplementedError

---
## 4 · Part C — does any of it beat the baseline?

This is the part that decides what you should actually do in your own project.

Ground rules, both taken straight from the lecture:

- **Scaffold split.** Random splits scatter analogues across train and test and
  inflate every number on the page.
- **The baseline is ECFP + random forest.** Every claim is measured against it.

In [ ]:
# --- Exercise 10: a scaffold split ---
# your code here!
#
# Write scaffold_split(df, frac_train=0.8, seed=SEED) that:
#   1. groups the row indices by df["scaffold"]
#   2. sorts the groups from largest to smallest (the standard deterministic
#      "scaffold split"; the big, well-populated series go into training)
#   3. fills the train set until it reaches frac_train of the data, then puts
#      everything else in test
#   4. returns (train_idx, test_idx) as numpy arrays
#
# Then build train/test index arrays and check that NO scaffold appears on both
# sides — that assertion is the whole point of the exercise.

raise NotImplementedError

In [ ]:
# --- Exercise 11: the head-to-head against the baseline ---
# your code here!
#
# On the SAME scaffold split, evaluate:
#   (a) ECFP4        + RandomForestRegressor(n_estimators=500, n_jobs=-1)
#   (b) ChemBERTa    + RidgeCV(alphas=np.logspace(-2, 4, 25))   <- the linear probe
#   (c) ChemBERTa    + RandomForestRegressor(n_estimators=500, n_jobs=-1)
#   (d) ECFP + ChemBERTa concatenated + RandomForest            <- do they add up?
#
# Report RMSE and R^2 for each in a small DataFrame, sorted by RMSE.
# Standardize the ChemBERTa features (fit the scaler on TRAIN only!).
#
# Write down your prediction before you run it.

raise NotImplementedError

Whatever you got: notice how *small* the differences are compared with the way
these methods are usually described. That is the Praski et al. (2025) result
reproduced on a single dataset in a single afternoon.

### 4.1 The key experiment — where is the crossover?

This is the notebook version of the lecture's most important figure. Full training
set, then progressively starve both models and watch what happens.

In [ ]:
# --- Exercise 12: the learning curve, and the crossover ---
# your code here!
#
# 1. For each n in SIZES = [50, 100, 200, 400, 800, 1600, 3000] (capped at the
#    training-set size), and for each of N_REPEATS = 3 random subsamples of the
#    TRAINING indices:
#       - fit ECFP + random forest  and  ChemBERTa + ridge  on that subsample
#       - evaluate both on the FULL (fixed) scaffold test set
# 2. Collect mean and standard deviation of the test RMSE per (model, n).
# 3. Plot RMSE vs n on a log x-axis, with error bars.
# 4. Report the smallest n at which the baseline overtakes the pre-trained model
#    (or state that it never does).
#
# Reuse the `evaluate` helper and the `rf()` factory you wrote in Exercise 11.
# Keep the test set fixed throughout — only the training subsample changes.
# Runtime: a couple of minutes.

raise NotImplementedError

In [ ]:
# --- Exercise 13: how much does the split type flatter you? ---
# your code here!
#
# Re-run the head-to-head from Exercise 11, but with a RANDOM 80/20 split instead
# of the scaffold split, and print the two sets of RMSEs side by side.
#
# The difference between the two columns is the amount of credit a random split
# would have handed you for free.

raise NotImplementedError

---
## 5 · Part D — where smooth representations break

Every representation we have used assumes that similar molecules have similar
properties. Activity cliffs are the counterexample, and they are exactly the
compounds a medicinal chemist cares about.

In [ ]:
# --- Exercise 14: hunt for cliffs ---
# your code here!
#
# 1. Using ECFP4 with Tanimoto, find every pair of molecules with similarity > 0.8.
#    (Reuse the BulkTanimotoSimilarity loop from Exercise 5, but keep the pairs.)
# 2. Among those, keep the pairs whose |Δ logD| > 1.0 — near-identical structures
#    with a tenfold difference in lipophilicity.
# 3. Report how many such pairs exist and what fraction of the high-similarity
#    pairs they represent.
# 4. Draw the most extreme pair with Draw.MolsToGridImage and look at what changed.

raise NotImplementedError

In [ ]:
# --- Exercise 15 (optional): how badly do the models do on the cliff pairs? ---
# your code here!
#
# Take the model you trained in Exercise 11 on the scaffold split. Compute its
# test RMSE on:
#    (a) the whole test set
#    (b) only those test molecules that appear in at least one cliff pair
#
# Compare. This is the van Tilborg et al. (2022) analysis in miniature, and it is
# the number you should report in any med-chem paper.

raise NotImplementedError

---
## 6 · Wrap-up

Write down your answers — we will compare across the room.

1. **How much of your ECFP variance did two principal components capture?** Would
   you be comfortable showing a PC1/PC2 plot as evidence that two chemotypes are
   distinct?
2. **Did the frozen ChemBERTa embeddings beat ECFP + random forest** on the full
   scaffold split? By how much, in log units? Is that difference larger than the
   spread you saw across repeats in the learning curve?
3. **Where was your crossover?** At what training-set size did the gap between the
   two curves become smaller than the error bars?
4. **How much did a random split flatter you?** Which of your two models benefited
   more from the leak — and can you explain why?
5. **Bring your own project to this.** How many labelled examples do you have? Which
   column of the "What to do on Monday" slide are you in?

### What to try next

- Swap `DeepChem/ChemBERTa-77M-MTR` for `DeepChem/ChemBERTa-77M-MLM` and re-run
  Exercises 8 and 11. The lecture claimed MTR beats MLM downstream — does it here?
- Use the `[CLS]` token instead of mean pooling. Usually worse; check.
- Replace ridge with a small MLP on the frozen embeddings. Does the extra capacity
  buy anything at n = 200?
- Concatenate 200 RDKit descriptors onto ECFP and re-run the baseline. Deng et al.
  found this combination hard to beat — see whether you agree.

### Where this goes next in the bootcamp

- **Thursday PM · Diffusion generative models** — the generative half of the story
  that VAEs lost.
- **Week 2, Thursday PM · Fine-tuning ChemBERTa** — the same model, but this time
  you unfreeze it and watch the learning rate.

---

*AI4Chemical Sciences Bootcamp 2026 · Caltech · jul@caltech.edu*